# Le Monde scraper

In [3]:
import requests
from bs4 import BeautifulSoup

response = requests.get("https://www.lemonde.fr/en/")
doc = BeautifulSoup(response.text, 'html.parser')

In [4]:
articles = doc.find_all(class_='article')

len(articles)

25

In [5]:
for article in articles:
    print('--------')

    # title
    print(article.select_one('.article__title').text.strip())

    # url
    url = article.select_one('a')['href']
    print(url)

    # subhead
    try:
        print(article.select_one('.article__desc').text.strip())
    except:
        print('no subhead found')

    # byline
    try: 
        print(article.select_one('.article__byline, .article__author-name').text.strip())
    except:
        print('no byline found')

    # premium
    # try:
    #     print(article.find(class_='sr-only').get_text())
    # except:
    #     print("Free to read")

    if article.find(class_='sr-only'):
        print("Subscribers only")
    else:
        print("Free to read")
    
    # type - assuming anything without explicit tag is news
    try:
        print(article.find(class_="article__type").text.strip())
    except:
        print("News")

    # image url
    img_tag = article.find('img')
    if img_tag:
        print(img_tag.get('src') or img_tag.get('data-src'))
    else:
        print("No image")

--------
From insider trading in France to an American cell phone: How Spain tracked down former prime minister Zapatero
https://www.lemonde.fr/en/les-decodeurs/article/2026/06/17/from-insider-trading-in-france-to-an-american-cell-phone-how-spain-tracked-down-former-prime-minister-zapatero_6754603_8.html
Judicial cooperation among several countries, including France and Switzerland, has helped uncover an international network relying on 'opaque financial channels,' which were allegedly used, among other things, to pay the former Socialist leader.
no byline found
Subscribers only
News
https://img.lemde.fr/2026/06/17/0/0/2656/1770/400/266/75/0/61e1c70_ftp-1-ejmstyyxgjrg-3dbd431db2f948f281505c8a122e8f3b-0-8050aa8c56224e4a8b3b3ff6cbacb7f2.jpg
--------
Trump and Iran's president sign deal to end the war
https://www.lemonde.fr/en/international/article/2026/06/18/trump-and-iran-s-president-sign-deal-to-end-the-war_6754610_4.html
no subhead found
no byline found
Free to read
News
https://img.l

In [7]:
rows = []

for article in articles:
    row = {}

    row['headline'] = article.select_one('.article__title').text.strip()
    row['url'] = article.select_one('a')['href']

    try:
        row['subhead'] = article.select_one('.article__desc').text.strip()
    except:
        row['subhead'] = None

    try: 
        row['byline'] = article.select_one('.article__byline, .article__author-name').text.strip()
    except:
        row['byline'] = None

    # if article.find(class_='sr-only'):
    #     row['premium'] = 'Subscribers only'
    # else:
    #     row['premium'] = 'Free to read'

    try:
        row['premium'] = article.find(class_='sr-only').get_text()
    except:
        row['premium'] = None
    
    try:
        row['article_type'] = article.find(class_="article__type").text.strip()
    except:
        row['article_type'] = None

    img_tag = article.find('img')
    if img_tag:
        row['img_url'] = img_tag.get('src') or img_tag.get('data-src')
    else:
        row['img_url'] = None
    
    rows.append(row)

rows

[{'headline': 'From insider trading in France to an American cell phone: How Spain tracked down former prime minister Zapatero',
  'url': 'https://www.lemonde.fr/en/les-decodeurs/article/2026/06/17/from-insider-trading-in-france-to-an-american-cell-phone-how-spain-tracked-down-former-prime-minister-zapatero_6754603_8.html',
  'subhead': "Judicial cooperation among several countries, including France and Switzerland, has helped uncover an international network relying on 'opaque financial channels,' which were allegedly used, among other things, to pay the former Socialist leader.",
  'byline': None,
  'premium': 'Subscribers only',
  'article_type': None,
  'img_url': 'https://img.lemde.fr/2026/06/17/0/0/2656/1770/400/266/75/0/61e1c70_ftp-1-ejmstyyxgjrg-3dbd431db2f948f281505c8a122e8f3b-0-8050aa8c56224e4a8b3b3ff6cbacb7f2.jpg'},
 {'headline': "Trump and Iran's president sign deal to end the war",
  'url': 'https://www.lemonde.fr/en/international/article/2026/06/18/trump-and-iran-s-presid

In [8]:
import pandas as pd

In [12]:
df = pd.DataFrame(rows)
# df = pd.json_normalize(rows) would be the same here since it's a flat dict
df.head()

,headline,url,subhead,byline,premium,article_type,img_url
0,From insider trading in France to an American ...,https://www.lemonde.fr/en/les-decodeurs/articl...,"Judicial cooperation among several countries, ...",NaN,Subscribers only,NaN,https://img.lemde.fr/2026/06/17/0/0/2656/1770/...
1,Trump and Iran's president sign deal to end th...,https://www.lemonde.fr/en/international/articl...,NaN,NaN,NaN,NaN,https://img.lemde.fr/2026/06/17/0/0/7949/5299/...
2,European Parliament's approval of new GMOs mar...,https://www.lemonde.fr/en/environment/article/...,NaN,NaN,Subscribers only,NaN,https://img.lemde.fr/2026/06/16/0/0/5119/3412/...
3,"Italian historian Carlo Ginzburg, founding fig...",https://www.lemonde.fr/en/obituaries/article/2...,Through his research on the 16th and 17th-cent...,NaN,Subscribers only,NaN,https://img.lemde.fr/2026/06/17/5/0/3872/2581/...
4,"Appointed by Trump to lead the Fed, Kevin Wars...",https://www.lemonde.fr/en/economy/article/2026...,"On Wednesday, June 17, the US central bank kep...",NaN,Subscribers only,NaN,https://img.lemde.fr/2026/06/17/1/0/7728/5152/...


In [13]:
df.to_csv('lemonde-scraped-results.csv', index=False)